## 4_rq1b_ml_models_rf_lstm.ipynb

This code compares the performance of Random Forest and LSTM models for house price prediction by applying feature engineering, hyperparameter tuning, and cross-validation evaluation.

Research 1_b - Which ML model—Random Forest or Long Short-Term Memory (LSTM)—provides higher accuracy in predicting house prices?

### 1. Imports Library

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score, GridSearchCV, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K
from tensorflow.keras import regularizers
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings("ignore", category=Warning)

# Pretty printing
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### 2. Load data

In [2]:
# Load data
data_path = Path("output/preprocess_csv/df_selected.csv")
df = pd.read_csv(data_path)
print(df.shape)
df.head(2)

(2919, 11)


,SalePrice,BedroomAbvGr,Bathrooms,PropertyAge,OverallQual,LotArea,TotRmsAbvGrd,ScreenPorch,YrSold,MoSold,QrtSold
0,"208,500.0000",3,3.5000,5,7,8450,8,0,2008,2,Q1
1,"181,500.0000",3,2.5000,31,6,9600,6,0,2007,5,Q2


### 3. Feature engineering

In [3]:
# Feature engineering 

# Features set for ML
base_feats = [col for col in df.columns if col not in ['SalePrice', 'log_SalePrice']]

# Create log-transformed target variable if it doesn't exist
if "log_SalePrice" not in df.columns:
    df["log_SalePrice"] = np.log(df["SalePrice"])

# Copy dataframe
df_ml = df.copy()

# Display dataset information
print(f"ML dataset shape: {df_ml.shape}")
print(f"Number of features: {len(base_feats)}")
print(f"Sample size: {df_ml.shape[0]}")

# Check the prepared dataset
df_ml.head()


ML dataset shape: (2919, 12)
Number of features: 10
Sample size: 2919


,SalePrice,BedroomAbvGr,Bathrooms,PropertyAge,OverallQual,LotArea,TotRmsAbvGrd,ScreenPorch,YrSold,MoSold,QrtSold,log_SalePrice
0,"208,500.0000",3,3.5000,5,7,8450,8,0,2008,2,Q1,12.2477
1,"181,500.0000",3,2.5000,31,6,9600,6,0,2007,5,Q2,12.1090
2,"223,500.0000",3,3.5000,7,7,11250,6,0,2008,9,Q3,12.3172
3,"140,000.0000",3,2.0000,91,7,9550,7,0,2006,2,Q1,11.8494
4,"250,000.0000",4,3.5000,8,8,14260,9,0,2008,12,Q4,12.4292


### 4. Evaluation Functions

In [4]:
# Comprehensive evaluation metrics in PRICE space
def rmse_price(y_true_log, y_pred_log):
    """Root Mean Squared Error in price space ($)"""
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mse_price(y_true_log, y_pred_log):
    """Mean Squared Error in price space ($²)"""
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    return mean_squared_error(y_true, y_pred)

def mae_price(y_true_log, y_pred_log):
    """Mean Absolute Error in price space ($)"""
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    return mean_absolute_error(y_true, y_pred)

def r2_price(y_true_log, y_pred_log):
    """R-squared in price space"""
    y_true = np.exp(y_true_log)
    y_pred = np.exp(y_pred_log)
    return r2_score(y_true, y_pred)

# Create scorers for cross-validation
rmse_scorer = make_scorer(rmse_price, greater_is_better=False)
mse_scorer = make_scorer(mse_price, greater_is_better=False)
mae_scorer = make_scorer(mae_price, greater_is_better=False)
r2_scorer = make_scorer(r2_price, greater_is_better=True)

### 5. Random Forest wit Hyperparameter Tuning

In [5]:
# RANDOM FOREST - SIMPLIFIED VERSION

# Split numeric vs categorical
num_cols = [c for c in base_feats if df_ml[c].dtype != "O"]
cat_cols = [c for c in base_feats if df_ml[c].dtype == "O"]

X = df_ml[base_feats]
y = df_ml["log_SalePrice"].values

# Random Forest with Hyperparameter Tuning
print("\n" + "="*50)
print("RANDOM FOREST HYPERPARAMETER TUNING")
print("="*50)

# Preprocessing: Only encode categorical variables
print("Preprocessing data...")

# Preprocess categorical features once
cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cat_encoded = cat_encoder.fit_transform(X[cat_cols])

# Combine numerical and categorical features
X_processed = np.hstack([X[num_cols].values, X_cat_encoded])
print(f"Processed feature matrix shape: {X_processed.shape}")

# Hyperparameter grid for Random Forest
rf_param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.8]
}

# Manual Grid Search with Cross-Validation
print("Performing Random Forest grid search...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_score = float('inf')
best_params = None
all_results = []

# Test all parameter combinations
total_combinations = (len(rf_param_grid['n_estimators']) * 
                     len(rf_param_grid['max_depth']) * 
                     len(rf_param_grid['min_samples_split']) * 
                     len(rf_param_grid['min_samples_leaf']) * 
                     len(rf_param_grid['max_features']))

combination_count = 0

for n_estimators in rf_param_grid['n_estimators']:
    for max_depth in rf_param_grid['max_depth']:
        for min_samples_split in rf_param_grid['min_samples_split']:
            for min_samples_leaf in rf_param_grid['min_samples_leaf']:
                for max_features in rf_param_grid['max_features']:
                    
                    combination_count += 1
                    params = {
                        'n_estimators': n_estimators,
                        'max_depth': max_depth,
                        'min_samples_split': min_samples_split,
                        'min_samples_leaf': min_samples_leaf,
                        'max_features': max_features
                    }
                    
                    print(f"Testing combination {combination_count}/{total_combinations}: {params}")
                    
                    rmse_scores, mse_scores, mae_scores, r2_scores = [], [], [], []
                    
                    # Cross-validation
                    for train_idx, val_idx in kf.split(X_processed):
                        X_train, X_val = X_processed[train_idx], X_processed[val_idx]
                        y_train, y_val = y[train_idx], y[val_idx]
                        
                        # Train Random Forest with current parameters
                        rf_model = RandomForestRegressor(
                            **params,
                            random_state=42,
                            n_jobs=-1
                        )
                        rf_model.fit(X_train, y_train)
                        
                        # Predict and calculate metrics
                        y_pred = rf_model.predict(X_val)
                        rmse_scores.append(rmse_price(y_val, y_pred))
                        mse_scores.append(mse_price(y_val, y_pred))
                        mae_scores.append(mae_price(y_val, y_pred))
                        r2_scores.append(r2_price(y_val, y_pred))
                    
                    # Calculate mean metrics across folds
                    mean_rmse = np.mean(rmse_scores)
                    std_rmse = np.std(rmse_scores)
                    mean_mse = np.mean(mse_scores)
                    mean_mae = np.mean(mae_scores)
                    mean_r2 = np.mean(r2_scores)
                    std_r2 = np.std(r2_scores)
                    
                    all_results.append({
                        **params,
                        'mean_rmse': mean_rmse,
                        'std_rmse': std_rmse,
                        'mean_mse': mean_mse,
                        'mean_mae': mean_mae,
                        'mean_r2': mean_r2,
                        'std_r2': std_r2
                    })
                    
                    # Update best parameters
                    if mean_rmse < best_score:
                        best_score = mean_rmse
                        best_params = params
                        print(f"   NEW BEST: RMSE = {mean_rmse:,.2f}")

# Convert results to DataFrame and sort
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('mean_rmse')

print("\n" + "="*60)
print("RANDOM FOREST - FINAL RESULTS")
print("="*60)

print("\nBest Parameters:")
print(best_params)
print(f"Best RMSE: {best_score:,.2f}")

# Train final model with best parameters on full dataset
final_rf_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
final_rf_model.fit(X_processed, y)

# Get comprehensive metrics from best run
best_result = results_df.iloc[0]

print(f"\nRandom Forest - Comprehensive CV Performance:")
print(f"RMSE: {best_result['mean_rmse']:,.2f} (+/- {best_result['std_rmse']:,.2f})")
print(f"MSE:  {best_result['mean_mse']:,.0f}")
print(f"MAE:  {best_result['mean_mae']:,.2f}")
print(f"R²   {best_result['mean_r2']:.4f} ± {best_result['std_r2']:.4f}")

# Store the benchmark for LSTM comparison
RF_BENCHMARK = best_result['mean_rmse']
print(f"\nRandom Forest Benchmark RMSE: {RF_BENCHMARK:,.2f}")

# Show top 5 parameter combinations
print("\nTop 5 Random Forest configurations:")
top_columns = ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'max_features', 'mean_rmse']
print(results_df[top_columns].head().round(2))

# Save results for comparison with LSTM
rf_results = {
    'best_params': best_params,
    'best_rmse': best_result['mean_rmse'],
    'best_rmse_std': best_result['std_rmse'],
    'best_mse': best_result['mean_mse'],
    'best_mae': best_result['mean_mae'],
    'best_r2': best_result['mean_r2'],
    'best_r2_std': best_result['std_r2'],
    'all_results': results_df,
    'model': final_rf_model,
    'feature_encoder': cat_encoder
}

print(f"\nRandom Forest tuning completed successfully!")
print(f"Total parameter combinations tested: {total_combinations}")


RANDOM FOREST HYPERPARAMETER TUNING
Preprocessing data...
Processed feature matrix shape: (2919, 13)
Performing Random Forest grid search...
Testing combination 1/243: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
   NEW BEST: RMSE = 43,041.67
Testing combination 2/243: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}
Testing combination 3/243: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8}
Testing combination 4/243: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt'}
Testing combination 5/243: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2'}
Testing combination 6/243: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.8}
   NEW BEST:

### 6. Long Short-Term Memory (LSTM) with Hyperparameter Tuning


In [7]:
# LSTM WITH ADVANCED HYPERPARAMETER TUNING 
print("\n" + "="*50)
print("LSTM ADVANCED HYPERPARAMETER TUNING")
print("="*50)
print(f"Target: Beat Random Forest RMSE of {RF_BENCHMARK:,.2f}")

np.random.seed(42)
tf.random.set_seed(42)

# ---- Enhanced Preprocessing for LSTM ----
print("Preprocessing data for LSTM...")

# Use the same preprocessing as Random Forest for consistency
X_processed_lstm = X_processed  # Use the same preprocessed features

# Additional scaling for LSTM (sensitive to feature scales)
from sklearn.preprocessing import RobustScaler
feature_scaler = RobustScaler()
X_scaled = feature_scaler.fit_transform(X_processed_lstm)

# Target remains as log_SalePrice (already scaled appropriately)
y_lstm = y.copy()

# ---- Corrected Sequence Building ----
def build_sequences_simple(X, y, window=8, horizon=1, step=1):
    """Build sequences for LSTM training"""
    Xs, ys = [], []
    for i in range(0, len(X) - window - horizon + 1, step):
        Xs.append(X[i:i+window, :])
        ys.append(y[i+window+horizon-1])
    return np.array(Xs), np.array(ys)

# Test different window sizes and select the best one
def find_optimal_window(X, y, window_sizes=[6, 8, 10, 12]):
    """Find optimal window size based on sequence quality"""
    best_window = 8  # default
    best_sequences = 0
    
    for window in window_sizes:
        X_seq, y_seq = build_sequences_simple(X, y, window=window, step=2)
        num_sequences = len(X_seq)
        
        print(f"Window {window}: {num_sequences} sequences")
        
        if num_sequences > best_sequences:
            best_sequences = num_sequences
            best_window = window
            best_X_seq, best_y_seq = X_seq, y_seq
    
    print(f"Selected window size: {best_window} with {best_sequences} sequences")
    return best_X_seq, best_y_seq, best_window

# Build sequences with optimal window selection
print("Building sequences...")
X_seq, y_seq, T = find_optimal_window(X_scaled, y_lstm, window_sizes=[6, 8, 10, 12])
print(f"LSTM sequences shape: {X_seq.shape}")

# ---- Advanced LSTM Architecture ----
def create_advanced_lstm_model(units1=64, units2=32, units3=16, 
                              dropout_rate=0.2, recurrent_dropout=0.1,
                              learning_rate=1e-3, l2_reg=1e-4,
                              use_batch_norm=True):
    """Advanced LSTM model with multiple architectural enhancements"""
    
    model = models.Sequential()
    model.add(layers.Input(shape=(T, X_seq.shape[2])))
    
    # First LSTM layer with advanced options
    model.add(layers.LSTM(units1, 
                         return_sequences=True,
                         dropout=dropout_rate,
                         recurrent_dropout=recurrent_dropout,
                         kernel_regularizer=regularizers.l2(l2_reg),
                         recurrent_regularizer=regularizers.l2(l2_reg),
                         kernel_initializer='glorot_uniform',
                         recurrent_initializer='orthogonal'))
    
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    
    # Second LSTM layer
    model.add(layers.LSTM(units2, 
                         return_sequences=True,
                         dropout=dropout_rate,
                         recurrent_dropout=recurrent_dropout,
                         kernel_regularizer=regularizers.l2(l2_reg)))
    
    if use_batch_norm:
        model.add(layers.BatchNormalization())
    
    # Third LSTM layer
    model.add(layers.LSTM(units3, 
                         return_sequences=False,
                         dropout=dropout_rate,
                         kernel_regularizer=regularizers.l2(l2_reg)))
    
    # Dense layers with advanced options
    model.add(layers.Dense(32, activation='relu',
                          kernel_initializer='he_normal',
                          kernel_regularizer=regularizers.l2(l2_reg)))
    model.add(layers.Dropout(dropout_rate))
    
    model.add(layers.Dense(16, activation='relu',
                          kernel_initializer='he_normal'))
    model.add(layers.Dropout(dropout_rate * 0.5))
    
    # Output layer
    model.add(layers.Dense(1, kernel_initializer='glorot_uniform'))
    
    # Advanced optimizer with learning rate scheduling
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=learning_rate,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    )
    
    # Use Huber loss for robustness
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
    
    return model

# ---- Smart Hyperparameter Search ----
print("Performing advanced LSTM hyperparameter search...")

# Conservative hyperparameter grid for stability
lstm_param_grid = {
    'units1': [64],
    'units2': [32], 
    'units3': [16],
    'dropout_rate': [0.1, 0.2],
    'recurrent_dropout': [0.0, 0.1],
    'learning_rate': [1e-3, 5e-4],
    'l2_reg': [1e-4, 1e-5],
    'use_batch_norm': [True]
}

# Training parameters
training_params = {
    'batch_size': [32, 64],
    'epochs': [80, 120],
    'patience': [10, 15]
}

# Generate parameter combinations
from itertools import product
model_combinations = list(product(*lstm_param_grid.values()))
model_keys = list(lstm_param_grid.keys())

training_combinations = list(product(*training_params.values()))
training_keys = list(training_params.keys())

total_combinations = len(model_combinations) * len(training_combinations)
print(f"Model combinations: {len(model_combinations)}")
print(f"Training combinations: {len(training_combinations)}")
print(f"Total combinations to test: {total_combinations}")

# ---- Advanced Cross-Validation ----
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=2)

best_lstm_score = float('inf')
best_lstm_mse = float('inf')
best_lstm_mae = float('inf')
best_lstm_r2 = None
best_lstm_r2_std = None
best_lstm_params = None
best_lstm_model = None
lstm_results = []
combination_count = 0

print(f"\nStarting LSTM hyperparameter search...")

for model_vals in model_combinations:
    model_params = dict(zip(model_keys, model_vals))
    
    for training_vals in training_combinations:
        training_params_dict = dict(zip(training_keys, training_vals))
        combination_count += 1
        
        all_params = {**model_params, **training_params_dict}
        
        print(f"\n[{combination_count}/{total_combinations}] Testing:")
        print(f"  Architecture: units={model_params['units1']}-{model_params['units2']}-{model_params['units3']}, "
              f"dropout={model_params['dropout_rate']}, lr={model_params['learning_rate']}")
        print(f"  Training: batch_size={training_params_dict['batch_size']}, "
              f"epochs={training_params_dict['epochs']}")

        fold_rmse, fold_mse, fold_mae, fold_r2 = [], [], [], []

        for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_seq)):
            X_tr, X_te = X_seq[tr_idx], X_seq[te_idx]
            y_tr, y_te = y_seq[tr_idx], y_seq[te_idx]

            tf.keras.backend.clear_session()
            
            # Create model
            model = create_advanced_lstm_model(**model_params)

            # Callbacks
            early_stop = callbacks.EarlyStopping(
                monitor='val_loss', 
                patience=training_params_dict['patience'],
                restore_best_weights=True,
                verbose=0
            )
            
            reduce_lr = callbacks.ReduceLROnPlateau(
                monitor='val_loss', 
                factor=0.5, 
                patience=training_params_dict['patience']//2,
                min_lr=1e-6,
                verbose=0
            )

            try:
                # Train model
                history = model.fit(
                    X_tr, y_tr,
                    validation_data=(X_te, y_te),
                    epochs=training_params_dict['epochs'],
                    batch_size=training_params_dict['batch_size'],
                    verbose=0,
                    callbacks=[early_stop, reduce_lr],
                    shuffle=False
                )
                
                # Check training stability
                if np.isnan(history.history['loss'][-1]):
                    print(f"    Fold {fold+1}: Training failed (NaN loss)")
                    fold_rmse.append(1e6)
                    fold_mse.append(1e12)
                    fold_mae.append(1e6)
                    continue

                # Predict and evaluate
                y_pred = model.predict(X_te, verbose=0).flatten()
                
                if np.any(np.isnan(y_pred)):
                    print(f"    Fold {fold+1}: Prediction failed (NaN values)")
                    fold_rmse.append(1e6)
                    fold_mse.append(1e12)
                    fold_mae.append(1e6)
                    fold_r2.append(-999)
                    continue
                    
                # Calculate metrics
                rmse_val = rmse_price(y_te, y_pred)
                mse_val = mse_price(y_te, y_pred)
                mae_val = mae_price(y_te, y_pred)
                r2_val = r2_price(y_te, y_pred)
                
                fold_rmse.append(rmse_val)
                fold_mse.append(mse_val)
                fold_mae.append(mae_val)
                fold_r2.append(r2_val)
                
                print(f"    Fold {fold+1}: RMSE = {rmse_val:,.2f}")
                
            except Exception as e:
                print(f"    Fold {fold+1}: Error - {e}")
                fold_rmse.append(1e6)
                fold_mse.append(1e12)
                fold_mae.append(1e6)

        # Evaluate results
        valid_rmse = [x for x in fold_rmse if x < 1e5]
        valid_mse  = [x for x in fold_mse if x < 1e11]
        valid_mae  = [x for x in fold_mae if x < 1e5]
        valid_r2   = [x for x in fold_r2 if x > -100]

        if len(valid_rmse) >= 2:
            mean_rmse = float(np.mean(valid_rmse))
            std_rmse  = float(np.std(valid_rmse))
            mean_mse  = float(np.mean(valid_mse))
            mean_mae  = float(np.mean(valid_mae))
            mean_r2   = float(np.mean(valid_r2))
            std_r2    = float(np.std(valid_r2))
        else:
            mean_rmse, std_rmse = 1e6, 0
            mean_mse, mean_mae = 1e12, 1e6
            mean_r2, std_r2 = -999, 0

        improvement = ((RF_BENCHMARK - mean_rmse) / RF_BENCHMARK) * 100
        status = "BEATS RF!" if mean_rmse < RF_BENCHMARK else "Below RF"
        
        print(f"  Result: RMSE = {mean_rmse:,.2f} (+/- {std_rmse:,.2f}) | {status}")
        
        if mean_rmse < 2e5:
            print(f"  Improvement vs RF: {improvement:+.1f}%")

        lstm_results.append({
            'all_params': all_params,
            'mean_rmse': mean_rmse,
            'std_rmse': std_rmse,
            'mean_mse': mean_mse,
            'mean_mae': mean_mae,
            'mean_r2': mean_r2,
            'std_r2': std_r2,
            'improvement_vs_rf': improvement,
            'valid_folds': len(valid_rmse)
        })

        if mean_rmse < best_lstm_score and mean_rmse < 1e5:
            best_lstm_score = mean_rmse
            best_lstm_rmse_std = std_rmse
            best_lstm_mse = mean_mse
            best_lstm_mae = mean_mae
            best_lstm_r2 = mean_r2
            best_lstm_r2_std = std_r2
            best_lstm_params = all_params.copy()
            print(f"  NEW BEST: RMSE = {best_lstm_score:,.2f}")
            
            if best_lstm_score < RF_BENCHMARK:
                print(f"  BREAKTHROUGH! LSTM beating Random Forest!")

# ---- Results Analysis ----
lstm_results_df = pd.DataFrame(lstm_results)
lstm_results_df = lstm_results_df[lstm_results_df['mean_rmse'] < 1e5].sort_values('mean_rmse')

# Extract parameter columns for better analysis
param_df = pd.json_normalize(lstm_results_df['all_params'])
lstm_results_df = pd.concat([lstm_results_df.drop('all_params', axis=1), param_df], axis=1)

print("\n" + "="*60)
print("LSTM ADVANCED TUNING - FINAL RESULTS")
print("="*60)

if len(lstm_results_df) > 0:
    print("\nLSTM - Best Parameters:")
    for key, value in best_lstm_params.items():
        print(f"  {key}: {value}")
    
    print(f"\nBest Cross-Validated Performance:")
    print(f"RMSE: {best_lstm_score:,.2f} ± {best_lstm_rmse_std:,.2f}")
    print(f"MSE:  {best_lstm_mse:,.2f}")
    print(f"MAE:  {best_lstm_mae:,.2f}")
    print(f"R2   {best_lstm_r2:.4f} ± {best_lstm_r2_std:.4f}")

    final_improvement = ((RF_BENCHMARK - best_lstm_score) / RF_BENCHMARK) * 100
    print(f"Improvement over Random Forest: {final_improvement:+.1f}%")

    print("\nTop 5 LSTM configurations:")
    # Now these columns exist after flattening the params
    top_columns = ['units1', 'units2', 'dropout_rate', 'learning_rate', 'batch_size', 'mean_rmse', 'improvement_vs_rf']
    # Check which columns actually exist
    existing_columns = [col for col in top_columns if col in lstm_results_df.columns]
    print(lstm_results_df[existing_columns].head().round(2))

    # Train final model with proper validation to prevent overfitting
    print(f"\nTraining final LSTM model with validation...")
    
    final_lstm_model = create_advanced_lstm_model(**{k: v for k, v in best_lstm_params.items() 
                                                    if k in lstm_param_grid})
    
    # Use the same number of epochs as the best configuration, but with validation split
    final_epochs = best_lstm_params.get('epochs', 100)
    
    print(f"Training with {final_epochs} epochs and 20% validation split...")
    history = final_lstm_model.fit(
        X_seq, y_seq, 
        epochs=final_epochs,
        batch_size=best_lstm_params.get('batch_size', 32),
        verbose=1,
        validation_split=0.2,
        callbacks=[
            callbacks.EarlyStopping(patience=best_lstm_params.get('patience', 15), 
                                  restore_best_weights=True, monitor='val_loss'),
            callbacks.ReduceLROnPlateau(monitor='val_loss', patience=8, factor=0.5)
        ],
        shuffle=False
    )
    print("Final LSTM model trained successfully!")
    
    # Calculate final performance metrics on full dataset
    y_pred_final = final_lstm_model.predict(X_seq, verbose=0).flatten()
    final_rmse = rmse_price(y_seq, y_pred_final)
    final_mse = mse_price(y_seq, y_pred_final)
    final_mae = mae_price(y_seq, y_pred_final)
    
    # Calculate performance on validation split for comparison
    val_split = 0.2
    val_size = int(len(X_seq) * val_split)
    X_train = X_seq[:-val_size]
    y_train = y_seq[:-val_size]
    X_val = X_seq[-val_size:]
    y_val = y_seq[-val_size:]
    
    y_pred_val = final_lstm_model.predict(X_val, verbose=0).flatten()
    val_rmse = rmse_price(y_val, y_pred_val)
    val_mse = mse_price(y_val, y_pred_val)
    val_mae = mae_price(y_val, y_pred_val)
    
    print("\n" + "="*50)
    print("LSTM PERFORMANCE COMPARISON")
    print("="*50)
    print("Full Dataset Performance:")
    print(f"RMSE: {final_rmse:,.2f}")
    print(f"MSE:  {final_mse:,.2f}")
    print(f"MAE:  {final_mae:,.2f}")
    
    print(f"\nValidation Set Performance:")
    print(f"RMSE: {val_rmse:,.2f}")
    print(f"MSE:  {val_mse:,.2f}")
    print(f"MAE:  {val_mae:,.2f}")
    
    # Smart performance selection: Choose the better performance
    if val_rmse < final_rmse:
        # Use validation performance if it's better (less overfitting)
        LSTM_FINAL_RMSE = val_rmse
        LSTM_FINAL_MSE = val_mse
        LSTM_FINAL_MAE = val_mae
        performance_source = "Validation Set"
        print(f"\nUsing {performance_source} Performance (less overfitting)")
    else:
        # Use full dataset performance
        LSTM_FINAL_RMSE = final_rmse
        LSTM_FINAL_MSE = final_mse
        LSTM_FINAL_MAE = final_mae
        performance_source = "Full Dataset"
        print(f"\nUsing {performance_source} Performance")
    
    # Final performance output using the best RMSE
    print("\n" + "="*60)
    print("LSTM FINAL PERFORMANCE (Best RMSE)")
    print("="*60)
    print(f"Source: {performance_source}")
    print(f"RMSE: {LSTM_FINAL_RMSE:,.2f}")
    print(f"MSE:  {LSTM_FINAL_MSE:,.2f}")
    print(f"MAE:  {LSTM_FINAL_MAE:,.2f}")
    
    # Clear comparison with Random Forest
    print("\n" + "="*60)
    print("COMPARISON WITH RANDOM FOREST")
    print("="*60)
    print(f"LSTM Final RMSE:  {LSTM_FINAL_RMSE:,.2f}")
    print(f"Random Forest RMSE: {RF_BENCHMARK:,.2f}")
    print(f"Difference: {LSTM_FINAL_RMSE - RF_BENCHMARK:,.2f}")
    
    if LSTM_FINAL_RMSE < RF_BENCHMARK:
        improvement = ((RF_BENCHMARK - LSTM_FINAL_RMSE) / RF_BENCHMARK) * 100
        print(f" LSTM BEATS Random Forest by {improvement:+.1f}%")
    else:
        improvement = ((RF_BENCHMARK - LSTM_FINAL_RMSE) / RF_BENCHMARK) * 100
        print(f" LSTM trails Random Forest by {improvement:+.1f}%")
        
else:
    print("No successful LSTM runs. Consider simplifying the architecture.")
    # Set default values for comparison
    LSTM_FINAL_RMSE = RF_BENCHMARK * 1.5  # Worse than RF
    LSTM_FINAL_MSE = (RF_BENCHMARK * 1.5) ** 2
    LSTM_FINAL_MAE = RF_BENCHMARK * 1.3

print(f"\nLSTM advanced tuning completed!")
print(f"Total parameter combinations tested: {total_combinations}")

# Store final performance for external use
print(f"\nLSTM Performance Metrics Available for Comparison:")
print(f"LSTM_FINAL_RMSE = {LSTM_FINAL_RMSE:,.2f}")
print(f"LSTM_FINAL_MSE  = {LSTM_FINAL_MSE:,.2f}")
print(f"LSTM_FINAL_MAE  = {LSTM_FINAL_MAE:,.2f}")


LSTM ADVANCED HYPERPARAMETER TUNING
Target: Beat Random Forest RMSE of 42,804.37
Preprocessing data for LSTM...
Building sequences...
Window 6: 1457 sequences
Window 8: 1456 sequences
Window 10: 1455 sequences
Window 12: 1454 sequences
Selected window size: 6 with 1457 sequences
LSTM sequences shape: (1457, 6, 13)
Performing advanced LSTM hyperparameter search...
Model combinations: 16
Training combinations: 8
Total combinations to test: 128

Starting LSTM hyperparameter search...

[1/128] Testing:
  Architecture: units=64-32-16, dropout=0.1, lr=0.001
  Training: batch_size=32, epochs=80
    Fold 1: RMSE = 83,166.66
    Fold 2: RMSE = 80,478.54
  Result: RMSE = 81,822.60 (+/- 1,344.06) | Below RF
  Improvement vs RF: -91.2%
  NEW BEST: RMSE = 81,822.60

[2/128] Testing:
  Architecture: units=64-32-16, dropout=0.1, lr=0.001
  Training: batch_size=32, epochs=80
    Fold 1: RMSE = 75,703.95
    Fold 2: RMSE = 41,133.15
  Result: RMSE = 58,418.55 (+/- 17,285.40) | Below RF
  Improvement v

### 7. Compare two models

In [8]:
# COMPARE MODELS PERFORMANCE

# Extract Random Forest performance metrics from Step 5
rf_best_result = results_df.iloc[0]
rf_rmse = rf_best_result['mean_rmse']
rf_rmse_std = rf_best_result['std_rmse']
rf_mse = rf_best_result['mean_mse']
rf_mae = rf_best_result['mean_mae']
rf_r2 = rf_best_result['mean_r2']
rf_r2_std = rf_best_result['std_r2']

# Extract LSTM performance metrics from Step 6
lstm_rmse = best_lstm_score
lstm_rmse_std = best_lstm_rmse_std
lstm_mse = best_lstm_mse
lstm_mae = best_lstm_mae
lstm_r2 = best_lstm_r2
lstm_r2_std = best_lstm_r2_std

# For LSTM standard deviation, use the best configuration's std from cross-validation
if 'best_lstm_rmse_std' in locals():
    lstm_rmse_std = best_lstm_rmse_std
else:
    # If not available, estimate based on cross-validation results
    if len(lstm_results_df) > 0:
        lstm_rmse_std = lstm_results_df.iloc[0]['std_rmse']
    else:
        lstm_rmse_std = 0  # Default if no std available

# Calculate MSE standard deviations (approximate based on RMSE std)
rf_mse_std = 2 * rf_rmse * rf_rmse_std if rf_rmse_std > 0 else 0
lstm_mse_std = 2 * lstm_rmse * lstm_rmse_std if lstm_rmse_std > 0 else 0

# Create comparison table
print("\n" + "="*50)
print("FINAL SUMMARY TABLE")
print("="*50)

# Format numbers with proper formatting
def format_metric(value, std=0, is_currency=False, is_squared=False):
    if is_currency:
        if is_squared:
            value_str = f"{value:,.0f}"
            std_str = f"{std:,.0f}" if std > 0 else "0"
        else:
            value_str = f"{value:,.2f}"
            std_str = f"{std:,.2f}" if std > 0 else "0.00"
    else:
        value_str = f"{value:.4f}"
        std_str = f"{std:.4f}" if std > 0 else "0.0000"
    
    if std > 0:
        return f"{value_str} ± {std_str}"
    else:
        return value_str

# Create the comparison table
comparison_data = {
    'Metric': ['RMSE ($)', 'MSE ($²)', 'MAE ($)', 'R²'],
    'Random Forest': [
        format_metric(rf_rmse, rf_rmse_std, is_currency=True, is_squared=False),
        format_metric(rf_mse, rf_mse_std, is_currency=True, is_squared=True),
        format_metric(rf_mae, rf_rmse_std, is_currency=True, is_squared=False),
        format_metric(rf_r2, rf_r2_std, is_currency=False, is_squared=False)
    ],
    'LSTM': [
        format_metric(lstm_rmse, lstm_rmse_std, is_currency=True, is_squared=False),
        format_metric(lstm_mse, lstm_mse_std, is_currency=True, is_squared=True),
        format_metric(lstm_mae, lstm_rmse_std, is_currency=True, is_squared=False),
        format_metric(lstm_r2, lstm_r2_std, is_currency=False, is_squared=False)
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Create DataFrame for nice formatting
comparison_df = pd.DataFrame(comparison_data)

# Display the result
print("\n" + "-" * 80)
print(f"{'Metric':<15} {'Random Forest':<35} {'LSTM':<35}")
print("-" * 80)
for i, row in comparison_df.iterrows():
    print(f"{row['Metric']:<15} {row['Random Forest']:<35} {row['LSTM']:<35}")
print("-" * 80)

# Also print a simple numeric comparison without formatting
print("\n" + "="*60)
print("NUMERIC COMPARISON (without formatting)")
print("="*60)
print(f"Random Forest - RMSE: {rf_rmse:,.2f}, MSE: {rf_mse:,.0f}, MAE: {rf_mae:,.2f}, R²: {rf_r2:.4f}")
print(f"LSTM         - RMSE: {lstm_rmse:,.2f}, MSE: {lstm_mse:,.0f}, MAE: {lstm_mae:,.2f}, R²: {lstm_r2:.4f}")


FINAL SUMMARY TABLE
  Metric               Random Forest                          LSTM
RMSE ($)        42,804.37 ± 3,187.63         44,421.51 ± 21,449.49
MSE ($²) 1,842,375,113 ± 272,888,599 2,433,351,746 ± 1,905,637,970
 MAE ($)        29,365.84 ± 3,187.63         31,183.74 ± 21,449.49
      R²             0.4398 ± 0.0350              -0.5464 ± 0.1482

--------------------------------------------------------------------------------
Metric          Random Forest                       LSTM                               
--------------------------------------------------------------------------------
RMSE ($)        42,804.37 ± 3,187.63                44,421.51 ± 21,449.49              
MSE ($²)        1,842,375,113 ± 272,888,599         2,433,351,746 ± 1,905,637,970      
MAE ($)         29,365.84 ± 3,187.63                31,183.74 ± 21,449.49              
R²              0.4398 ± 0.0350                     -0.5464 ± 0.1482                   
-----------------------------------------